# Spectral Analysis

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import signal
from scipy import stats
from scipy import ndimage
from tqdm import tqdm
import matplotlib.pyplot as plt
import importlib

from pxg import cbor
importlib.reload(cbor)

from pxg.cbor import CborDatabase, CborRecord

from vta.dsp import SignalFilter, LYN_WIND, MED_WIND, LYN_HIGH
from vta.fft import FFT, Spectral, FFT_FREQS, FFT_WIN

FFT_RES = FFT_WIN % 1000
FFT_SEC = FFT_WIN - FFT_RES

%config InlineBackend.figure_format = 'retina'

In [ ]:
### DATA PROCESSING ###

DDIR = "./data"

def Frame(db: str, rid: str, segs: int) -> pd.DataFrame:
    return pd.DataFrame({
        "DB":   [db] * segs,
        "RID":  [rid] * segs,
        "SEG":  np.zeros(segs, dtype=np.int32),
        "Time": np.zeros(segs, dtype=np.int32),
        "End":  np.zeros(segs, dtype=np.int32),
        "Epis": np.zeros(segs, dtype=np.int16),
        "Beat": np.zeros(segs, dtype=np.int16),

        # Episodes
        "NSR":  np.zeros(segs, dtype=np.int16),
        "BGM":  np.zeros(segs, dtype=np.int16),
        "TGM":  np.zeros(segs, dtype=np.int16),
        "VTH":  np.zeros(segs, dtype=np.int16),
        "VFL":  np.zeros(segs, dtype=np.int16),
        "VFN":  np.zeros(segs, dtype=np.int16),
        "VFB":  np.zeros(segs, dtype=np.int16),
        "AFL":  np.zeros(segs, dtype=np.int16),
        "AFB":  np.zeros(segs, dtype=np.int16),
        "EPX":  np.zeros(segs, dtype=np.int16),

        # Annotations
        # Group N: N, L, R, B
        "N":    np.zeros(segs, dtype=np.int16),
        "L":    np.zeros(segs, dtype=np.int16),
        "R":    np.zeros(segs, dtype=np.int16),
        "B":    np.zeros(segs, dtype=np.int16),
        # Group A: A
        "A":    np.zeros(segs, dtype=np.int16),
        # Group S: S
        "S":    np.zeros(segs, dtype=np.int16),
        # Group C: J, a, j, e, n
        "C":    np.zeros(segs, dtype=np.int16),
        # Group V: V, r, E
        "V":    np.zeros(segs, dtype=np.int16),
        "W":    np.zeros(segs, dtype=np.int16),
        # Group F: F
        "F":    np.zeros(segs, dtype=np.int16),
        # Group Q: Q, f
        "Q":    np.zeros(segs, dtype=np.int16),
        # Group P: /
        "P":    np.zeros(segs, dtype=np.int16),
        # Group O: ~, !, [, ]
        "O":    np.zeros(segs, dtype=np.int16),
        # Group Z: Other
        "Z":    np.zeros(segs, dtype=np.int16),
        # Group X: Unclassified
        "X":    np.zeros(segs, dtype=np.int16),

        # Data
        # "RAW":  pd.array([None] * segs, dtype=object),
        "FFT":  pd.array([None] * segs, dtype=object),

        "Target": [""] * segs,

        "LOFB":  np.zeros(segs, dtype=np.float64), # Low-frequency band power (0.5-3.5 Hz) / (LOFB + MDFB + HIFB)
        "MDFB":  np.zeros(segs, dtype=np.float64), # Mid-frequency band power (3.5-8 Hz) / (LOFB + MDFB + HIFB)
        "HIFB":  np.zeros(segs, dtype=np.float64), # High-frequency band power (8-20 Hz) / (LOFB + MDFB + HIFB)
        "OTFB":  np.zeros(segs, dtype=np.float64), # Other-frequency band power (1.5-24 Hz) / (LOFB + MDFB + HIFB)
        
        "DOMF":  np.zeros(segs, dtype=np.float64), # Dominant frequency (Hz)
        "CENF":  np.zeros(segs, dtype=np.float64), # Centorid of the spectrum (Hz) = SUM(Freq * Power) / SUM(Power)
        "SPRD":  np.zeros(segs, dtype=np.float64), # Spread of the spectrum (Hz) = sqrt(SUM((Freq - CEN)^2 * Power) / SUM(Power))
        "GINI":  np.zeros(segs, dtype=np.float64), # Gini index of the spectrum
        "ENTR":  np.zeros(segs, dtype=np.float64), # Shannon entropy of the spectrum
        "FLAT":  np.zeros(segs, dtype=np.float64), # Spectral flatness

    })
pass #def

class Segment:
    def __init__(self, df: pd.DataFrame, ix: int):
        self.df = df
        self.ix = ix

        self.Epis: list[cbor.CborAnnotation] = []
        self.Anns: list[cbor.CborAnnotation] = []

        # self.Dominant = 0.00
        # self.Centroid = 0.00
        # self.Spread = 0.00
        # self.Ginidex = 0.0000
        # self.Entropy = 0.0000
        # self.Flatness = 0.0000
    pass #def

    @property
    def DB(self) -> int:
        return self.df.at[self.ix, "DB"] # type: ignore
    pass #property

    @property
    def RID(self) -> int:
        return self.df.at[self.ix, "RID"] # type: ignore
    pass #property

    @property
    def SEG(self) -> int:
        return self.df.at[self.ix, "SEG"] # type: ignore
    pass #property

    @property
    def Time(self) -> int:
        return self.df.at[self.ix, "Time"] # type: ignore
    pass #property

    @property
    def End(self) -> int:
        return self.df.at[self.ix, "End"] # type: ignore
    pass #property

    # // EPISODES //////////////////

    @property
    def NSR(self) -> int:
        return self.df.at[self.ix, "NSR"] # type: ignore
    @NSR.setter
    def NSR(self, value: int):
        self.df.at[self.ix, "NSR"] = value
    pass #property

    @property
    def BGM(self) -> int:
        return self.df.at[self.ix, "BGM"] # type: ignore
    @BGM.setter
    def BGM(self, value: int):
        self.df.at[self.ix, "BGM"] = value
    pass #property

    @property
    def TGM(self) -> int:
        return self.df.at[self.ix, "TGM"] # type: ignore
    @TGM.setter
    def TGM(self, value: int):
        self.df.at[self.ix, "TGM"] = value
    pass #property

    @property
    def VTH(self) -> int:
        return self.df.at[self.ix, "VTH"] # type: ignore
    @VTH.setter
    def VTH(self, value: int):
        self.df.at[self.ix, "VTH"] = value
    pass #property

    @property
    def VFL(self) -> int:
        return self.df.at[self.ix, "VFL"] # type: ignore
    @VFL.setter
    def VFL(self, value: int):
        self.df.at[self.ix, "VFL"] = value
    pass #property

    @property
    def VFN(self) -> int:
        return self.df.at[self.ix, "VFN"] # type: ignore
    @VFN.setter
    def VFN(self, value: int):
        self.df.at[self.ix, "VFN"] = value
    pass #property

    @property
    def VFB(self) -> int:
        return self.df.at[self.ix, "VFB"] # type: ignore
    @VFB.setter
    def VFB(self, value: int):
        self.df.at[self.ix, "VFB"] = value
    pass #property

    @property
    def AFL(self) -> int:
        return self.df.at[self.ix, "AFL"] # type: ignore
    @AFL.setter
    def AFL(self, value: int):
        self.df.at[self.ix, "AFL"] = value
    pass #property

    @property
    def AFB(self) -> int:
        return self.df.at[self.ix, "AFB"] # type: ignore
    @AFB.setter
    def AFB(self, value: int):
        self.df.at[self.ix, "AFB"] = value
    pass #property

    @property
    def EPX(self) -> int:
        return self.df.at[self.ix, "EPX"] # type: ignore
    @EPX.setter
    def EPX(self, value: int):
        self.df.at[self.ix, "EPX"] = value
    pass #property

    # // Group N: N, L, R, B ///////
    
    @property
    def N(self) -> int:
        return self.df.at[self.ix, "N"] # type: ignore
    @N.setter
    def N(self, value: int):
        self.df.at[self.ix, "N"] = value
    pass #property

    @property
    def L(self) -> int:
        return self.df.at[self.ix, "L"] # type: ignore
    @L.setter
    def L(self, value: int):
        self.df.at[self.ix, "L"] = value
    pass #property

    @property
    def R(self) -> int:
        return self.df.at[self.ix, "R"] # type: ignore
    @R.setter
    def R(self, value: int):
        self.df.at[self.ix, "R"] = value
    pass #property

    @property
    def B(self) -> int:
        return self.df.at[self.ix, "B"] # type: ignore
    @B.setter
    def B(self, value: int):
        self.df.at[self.ix, "B"] = value
    pass #property

    # // Group A: A ///////////////

    @property
    def A(self) -> int:
        return self.df.at[self.ix, "A"] # type: ignore
    @A.setter
    def A(self, value: int):
        self.df.at[self.ix, "A"] = value
    pass #property

    # // Group S: S ////////////////

    @property
    def S(self) -> int:
        return self.df.at[self.ix, "S"] # type: ignore
    @S.setter
    def S(self, value: int):
        self.df.at[self.ix, "S"] = value
    pass #property

    # // Group C: J, a, j, e, n ///

    @property
    def C(self) -> int:
        return self.df.at[self.ix, "C"] # type: ignore
    @C.setter
    def C(self, value: int):
        self.df.at[self.ix, "C"] = value
    pass #property

    # // Group V: V, r, E //////////

    @property
    def V(self) -> int:
        return self.df.at[self.ix, "V"] # type: ignore
    @V.setter
    def V(self, value: int):
        self.df.at[self.ix, "V"] = value
    pass #property

    @property
    def W(self) -> int:
        return self.df.at[self.ix, "W"] # type: ignore
    @W.setter
    def W(self, value: int):
        self.df.at[self.ix, "W"] = value
    pass #property

    # // Group F: F /////////////////

    @property
    def F(self) -> int:
        return self.df.at[self.ix, "F"] # type: ignore
    @F.setter
    def F(self, value: int):
        self.df.at[self.ix, "F"] = value
    pass #property

    # // Group Q: Q, f ///////////////

    @property
    def Q(self) -> int:
        return self.df.at[self.ix, "Q"] # type: ignore
    @Q.setter
    def Q(self, value: int):
        self.df.at[self.ix, "Q"] = value
    pass #property

    # // Group P: / //////////////////

    @property
    def P(self) -> int:
        return self.df.at[self.ix, "P"] # type: ignore
    @P.setter
    def P(self, value: int):
        self.df.at[self.ix, "P"] = value
    pass #property

    # // Group O: ~, !, [, ] //////////

    @property
    def O(self) -> int:
        return self.df.at[self.ix, "O"] # type: ignore
    @O.setter
    def O(self, value: int):
        self.df.at[self.ix, "O"] = value
    pass #property

    # // Group Z ///////////////////////

    @property
    def Z(self) -> int:
        return self.df.at[self.ix, "Z"] # type: ignore
    @Z.setter
    def Z(self, value: int):
        self.df.at[self.ix, "Z"] = value
    pass #property

    # // Group X //////////////////////

    @property
    def X(self) -> int:
        return self.df.at[self.ix, "X"] # type: ignore
    @X.setter
    def X(self, value: int):
        self.df.at[self.ix, "X"] = value
    pass #property

    # // Signal //////////////////

    @property
    def RAW(self) -> np.ndarray:
        return self.df.at[self.ix, "RAW"] # type: ignore
    @RAW.setter
    def RAW(self, value: np.ndarray):
        self.df.at[self.ix, "RAW"] = value # type: ignore
    pass #property

    @property
    def FFT(self) -> np.ndarray:
        return self.df.at[self.ix, "FFT"] # type: ignore
    @FFT.setter
    def FFT(self, value: np.ndarray):
        self.df.at[self.ix, "FFT"] = value # type: ignore
    pass #property

    ## // Spectral /////////////////

    @property
    def LOFB(self) -> float:
        return self.df.at[self.ix, "LOFB"] # type: ignore
    @LOFB.setter
    def LOFB(self, value: float):
        self.df.at[self.ix, "LOFB"] = value
    pass #property

    @property
    def MDFB(self) -> float:
        return self.df.at[self.ix, "MDFB"] # type: ignore
    @MDFB.setter
    def MDFB(self, value: float):
        self.df.at[self.ix, "MDFB"] = value
    pass #property

    @property
    def HIFB(self) -> float:
        return self.df.at[self.ix, "HIFB"] # type: ignore
    @HIFB.setter
    def HIFB(self, value: float):
        self.df.at[self.ix, "HIFB"] = value
    pass #property

    @property
    def OTFB(self) -> float:
        return self.df.at[self.ix, "OTFB"] # type: ignore
    @OTFB.setter
    def OTFB(self, value: float):
        self.df.at[self.ix, "OTFB"] = value
    pass #property

    @property
    def DOMF(self) -> float:
        return self.df.at[self.ix, "DOMF"] # type: ignore
    @DOMF.setter
    def DOMF(self, value: float):
        self.df.at[self.ix, "DOMF"] = value
    pass #property

    @property
    def CENF(self) -> float:
        return self.df.at[self.ix, "CENF"] # type: ignore
    @CENF.setter
    def CENF(self, value: float):
        self.df.at[self.ix, "CENF"] = value
    pass #property

    @property
    def SPRD(self) -> float:
        return self.df.at[self.ix, "SPRD"] # type: ignore
    @SPRD.setter
    def SPRD(self, value: float):
        self.df.at[self.ix, "SPRD"] = value
    pass #property

    @property
    def ENTR(self) -> float:
        return self.df.at[self.ix, "ENTR"] # type: ignore
    @ENTR.setter
    def ENTR(self, value: float):
        self.df.at[self.ix, "ENTR"] = value
    pass #property

    @property
    def GINI(self) -> float:
        return self.df.at[self.ix, "GINI"] # type: ignore
    @GINI.setter
    def GINI(self, value: float):
        self.df.at[self.ix, "GINI"] = value
    pass #property

    @property
    def FLAT(self) -> float:
        return self.df.at[self.ix, "FLAT"] # type: ignore
    @FLAT.setter
    def FLAT(self, value: float):
        self.df.at[self.ix, "FLAT"] = value
    pass #property

    def Attach(self, epis: list[cbor.CborAnnotation], anns: list[cbor.CborAnnotation]) -> 'Segment':
        self.Epis = epis
        self.Anns = anns

        # if rec: self.RAW = rec.Signal[self.Time:self.End]

        for e in epis:
            a = max(self.Time, e.Time)
            b = min(self.End, e.End)
            k = max(0, b - a)
            if e.Type == "[":
                self.VFN += k
            elif e.Note == "(N":
                self.NSR += k
            elif e.Note == "(B":
                self.BGM += k
            elif e.Note == "(T":
                self.TGM += k
            elif e.Note == "(VT":
                self.VTH += k
            elif e.Note == "(VFL":
                self.VFL += k
            elif e.Note == "(VF":
                self.VFB += k
            elif e.Note == "(AFL":
                self.AFL += k
            elif e.Note == "(AFIB":
                self.AFB += k
            else:
                self.EPX += k
            pass #if
        pass #for

        for a in anns:
            if a.Type == "N":
                self.N += 1
            elif a.Type == "L":
                self.L += 1
            elif a.Type == "R":
                self.R += 1
            elif a.Type == "B":
                self.B += 1
            elif a.Type == "A":
                self.A += 1
            elif a.Type == "S":
                self.S += 1
            elif a.Type in ["J", "a", "j", "e", "n"]:
                self.C += 1
            elif a.Type == "V":
                self.V += 1
            elif a.Type in ["r", "E"]:
                self.W += 1
            elif a.Type == "F":
                self.F += 1
            elif a.Type in ["Q", "f"]:
                self.Q += 1
            elif a.Type == "/":
                self.P += 1
            elif a.Type in ["~", "!", "[", "]"]:
                self.O += 1
            else:
                self.Z += 1
            pass #if
        pass #for
        
        return self
    pass #def
pass #class

PARQUET = True

def Load(db: str, rid: str) -> pd.DataFrame:
    path = os.path.join(DDIR, db)
    if PARQUET:
        df = pd.read_parquet(os.path.join(path, f'{rid}.pqt'))
    else:
        df = pd.read_csv(os.path.join(path, f'{rid}.tsv'), sep='\t')
        df['FFT'] = list(np.load(os.path.join(path, f'{rid}.npy')))  # restore array per row
    pass #if
    return df
pass #def

def Save(df: pd.DataFrame, db: str, rid: str):
    path = os.path.join(DDIR, db)
    os.makedirs(path, exist_ok=True)
    if PARQUET:
        df.to_parquet(os.path.join(path, f'{rid}.pqt'), compression='snappy', index=False)
    else:
        np.save(os.path.join(path, f'{rid}.npy'), np.stack(df['FFT'].values)) # type: ignore
        df.drop(columns=['FFT']).to_csv(os.path.join(path, f'{rid}.tsv'), sep='\t', index=False)
    pass #if
pass #def

In [ ]:
### CLASSIFICATION ###

def Classify(df: pd.DataFrame, THR = 900):
    df["Target"] = "MIX"
    for ix, row in df.iterrows():
        ALL = row["VTH"] + row["VFL"] + row["VFN"] + row["VFB"] + row["AFL"] + row["AFB"] + row["BGM"] + row["TGM"]
        if ALL == 0: 
            df.at[ix, "Target"] = "NSR"
            continue
        pass #if
        if row["AFL"] > THR:
            t = "AFL"
        elif row["AFB"] > THR:
            t = "AFB"
        elif row["BGM"] > THR:
            t = "BGM"
        elif row["TGM"] > THR:
            t = "TGM"
        elif row["VTH"] > THR:
            t = "VTH"
        elif row["VFL"] > THR:
            t = "VFL"
        elif row["VFN"] > THR:
            t = "VFL"
        elif row["VFB"] > THR:
            t = "VFB"
        else:
            t = "MIX"
        pass #if
        df.at[ix, "Target"] = t
    pass #for

    for ix in df.index:
        sp = Spectral(df.at[ix, "FFT"]) # type: ignore

        df.at[ix, "DOMF"] = sp.Prominent
        df.at[ix, "LOFB"] = sp.LowBand
        df.at[ix, "MDFB"] = sp.MidBand
        df.at[ix, "HIFB"] = sp.HighBand
        df.at[ix, "OTFB"] = sp.OutBand
        df.at[ix, "CENF"] = sp.Centroid
        df.at[ix, "SPRD"] = sp.Spread
        df.at[ix, "ENTR"] = sp.Entropy
        df.at[ix, "GINI"] = sp.Gini
        df.at[ix, "FLAT"] = sp.Flatness
    pass #for
pass #def

In [ ]:
### PROCESSING ###

def ProcessRecord(rec: CborRecord, STEP = 500) -> pd.DataFrame:
    rem = len(rec.Signal) % 1000
    if rem < 24:
        rec.Signal = np.append(
            rec.Signal,
            np.zeros(24 - rem, dtype=rec.Signal.dtype)
        )
    pass #if
    rec.Signal = SignalFilter(rec.Signal, L = LYN_WIND, M = MED_WIND, H = LYN_HIGH * 0)

    anns = rec.Annotations
    size = len(rec.Signal) # rec.Info.Channels[rec.Best].Size
    snum = size // STEP - 1

    six = 0
    aix = 0
    epis = []
    beat = []
    frame = Frame(rec.DB, rec.RID, snum)

    for t in range(STEP, size, STEP):
        while aix < len(anns) and anns[aix].Time < t:
            if anns[aix].Type == "[" or anns[aix].Type == "+" and anns[aix].Note[:1] == "(":
                # print(anns[aix], anns[aix].End)
                epis.append(anns[aix])
            else:
                beat.append(anns[aix])
            pass #if
            aix += 1
        pass #while
        c = t - FFT_SEC
        r = 0
        while r < len(beat) and beat[r].Time < c:
            r += 1
        pass #while
        beat = beat[r:]
        r = 0
        while r < len(epis) and epis[r].End <= c:
            r += 1
        pass #while
        epis = epis[r:]
        if c < 0: continue

        seg = rec.Signal[c:t+FFT_RES]

        frame.at[six, "SEG"] = six
        frame.at[six, "Time"] = c
        frame.at[six, "End"] = t
        frame.at[six, "Epis"] = len(epis)
        frame.at[six, "Beat"] = len(beat)
        
        # frame.at[six, "RAW"] = seg # type: ignore
        frame.at[six, "FFT"] = FFT(seg) # type: ignore
        
        # print(six+1, c, t+FFT_RES, len(epis), len(beat), len(seg))
        # ProcessSegment(seg, epis, beat)

        Segment(frame, six).Attach(epis, beat)
        six += 1
    pass #for

    return frame
pass #def

def ProcessDatabase(name: str, regx: str = ""):
    db = CborDatabase(name)
    path = os.path.join(DDIR, name)
    os.makedirs(path, exist_ok=True)

    with open(os.path.join(path, f'RECORDS'), 'w') as fr:
        n = (name + " " * 10)[:8]
        for rid in tqdm(db.Records, ncols=120, desc=f'= {n}'):
            if regx and not re.search(regx, rid): continue
            rec = db.Load(rid)
            frame = ProcessRecord(rec)
            Save(frame, rec.DB, rec.RID)
            fr.write(f'{rid}\n')
        pass #for
    pass #with

    print()
pass #def

def LoadRecord(db: str, rid: str) -> pd.DataFrame:
    df = Load(db, rid)
    Classify(df)
    return df
pass #def

def LoadDatabase(name: str, save=False) -> pd.DataFrame:
    path = os.path.join(DDIR, name)
    with open(os.path.join(path, f'RECORDS'), 'r') as f:
        rids = [line.strip() for line in f]
    pass #with
    frames = []
    n = (name + " " * 10)[:8]
    for rid in tqdm(rids, ncols=120, desc=f'> {n}'):
        df = LoadRecord(name, rid)
        frames.append(df)
    pass #for

    df = pd.concat(frames)
    df.reset_index(drop=True, inplace=True)

    # print(len(frames))
    # display(frames.head())

    print()

    # Count by Target
    counts = df["Target"].value_counts()
    for label, count in counts.items():
        print(f"{label}: {count}")
    pass #for

    print()

    if save: Save(df, 'db', name)

    return df
pass #def

In [ ]:
### VISUALIZATION ###

def PlotRecord(rf: pd.DataFrame, rid: str):
    plt.figure(figsize=(21, 3))
    plt.title(f"{rid}")
    plt.ylim(-6, 12)

    t = rf["Time"] / 250
    plt.xlim(0, t.max())

    plt.plot(t, rf["DOMF"], color="green", alpha=0.5)

    plt.plot(t, rf["CENF"], color="blue", alpha=0.5, linewidth=0.7)
    # plt.plot(t, rf["Spread"], color="red", alpha=0.5, linewidth=0.5, linestyle="--")

    plt.axhspan(10, 12, color="gray", alpha=0.1)
    plt.plot(t, rf["ENTR"] * 4 + 10 - 2, color="black", alpha=0.7, linewidth=0.7)
    # R = rf["MFB"] / (rf["HFB"] + rf["MFB"] + 1e-10)
    # plt.plot(t, R * 10, color="black", alpha=0.7)

    base = -6
    mul = 4
    WFB = (rf["LOFB"] + rf["MDFB"] + rf["HIFB"]) / mul
    # plot stacked area for LFB/MFB/HFB
    plt.fill_between(t, base + 0, base + rf["LOFB"] / WFB, color="gray", alpha=0.3)
    plt.fill_between(t, base + rf["LOFB"] / WFB, base + (rf["LOFB"] + rf["MDFB"]) / WFB, color="blue", alpha=0.3)
    plt.fill_between(t, base + (rf["LOFB"] + rf["MDFB"]) / WFB, base + mul, color="purple", alpha=0.3)
    # plt.plot(t, rf["DFB"] / rf["OFB"] * mul + base, color="magenta", alpha=0.7, linewidth=0.7)

    base = -2
    mul = 2
    plt.axhspan(base, 0, color="gray", alpha=0.1)
    vth = rf["VTH"] / 1000 * mul + base
    vfl = rf["VFL"] / 1000 * mul + base
    vfn = rf["VFN"] / 1000 * mul + base
    vfb = rf["VFB"] / 1000 * mul + base

    # plt.plot(t, vth, color="green", alpha=0.7)
    plt.fill_between(t, base, vth, color="green", alpha=0.15)

    # plt.plot(t, vfl, color="orange", alpha=0.7)
    plt.fill_between(t, base, vfl, color="orange", alpha=0.2)

    # plt.plot(t, vfn, color="orange", alpha=0.7)
    plt.fill_between(t, base, vfn, color="orange", alpha=0.2)

    # plt.plot(t, vfb, color="magenta", alpha=0.7)
    plt.fill_between(t, base, vfb, color="magenta", alpha=0.15)

    plt.show()
pass #def

def PlotDatabase(db: str):
    # rf[rf["RID"] == rid]
    path = os.path.join(DDIR, db)
    with open(os.path.join(path, f'RECORDS'), 'r') as f:
        rids = [line.strip() for line in f]
    pass #with
    for rid in rids:
        df = LoadRecord(db, rid)
        PlotRecord(df, rid)
    pass #for
pass #def

In [ ]:
print()

# ProcessDatabase('mitdb')
ProcessDatabase('cudb')
# ProcessDatabase('vfdb')
# ProcessDatabase('ahadb')

print()

# mitdb = LoadDatabase('mitdb')
# cudb = LoadDatabase('cudb')
# ahadb = LoadDatabase('ahadb')
# vfdb = LoadDatabase('vfdb')

# PlotDatabase('mitdb')
PlotDatabase('cudb')
# PlotDatabase(vfdb)
# PlotDatabase(ahadb)

pass